In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# CREATE DATASET FOR LIGHTGBM (LAGGED FEATURES)
# ==========================================

def create_dataset(data, time_step=10):
    """
    Transforms time series data into a supervised learning dataset with lagged features
    for LightGBM.

    Args:
        data (np.array): Input time series data. Expected shape (n_samples, n_features).
        time_step (int): Number of previous time steps to use as input features (lags).

    Returns:
        tuple: (X, y) where X is the feature matrix (lagged values) and y is the target.
    """
    X, y = [], []
    num_features = data.shape[1]

    for i in range(len(data) - time_step):
        # Flatten the previous 'time_step' observations into a single row for X
        # Each observation has 'num_features'
        flattened_features = data[i:(i + time_step)].flatten()
        X.append(flattened_features)
        # The target 'y' is the 'revenue' (first feature, index 0) of the next time step
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


# ==========================================
# RMSSE
# ==========================================

def rmsse_score(y_true, y_pred):
    """
    Calculates the Root Mean Squared Scaled Error (RMSSE).
    """
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    diff = np.diff(y_true)
    scale = np.sqrt(np.mean(diff**2))
    if scale == 0:
        return rmse
    return rmse / scale

# NOTE: The original CNN model building and training functions (`build_best_cnn`, `train_cnn`)
# have been removed as per user request to switch to LightGBM.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("aryayadav0513/m5-forecasting-accuracy")
print("Path:", path)

100%|██████████| 45.8M/45.8M [00:00<00:00, 118MB/s]

Extracting files...


Path: /root/.cache/kagglehub/datasets/aryayadav0513/m5-forecasting-accuracy/versions/1


In [ ]:
import os

os.listdir(path)

['m5-forecasting-accuracy']

In [ ]:
#os.environ['TF_DETERMINISTIC_OPS'] = '1'
#os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

#tf.config.experimental.enable_op_determinism()

In [ ]:
os.listdir(f"{path}/m5-forecasting-accuracy")

['sell_prices.csv',
 'sample_submission.csv',
 'calendar.csv',
 'sales_train_evaluation.csv',
 'sales_train_validation.csv']

In [ ]:
import pandas as pd

In [ ]:
sales = pd.read_csv(f"{path}/m5-forecasting-accuracy/sales_train_validation.csv")
calendar = pd.read_csv(f"{path}/m5-forecasting-accuracy/calendar.csv")
prices = pd.read_csv(f"{path}/m5-forecasting-accuracy/sell_prices.csv")

In [ ]:
import os

dataset_path = os.path.join(path, "m5-forecasting-accuracy")

sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:
print(sales.shape)
print(calendar.shape)
print(prices.shape)

(30490, 1919)
(1969, 14)
(6841121, 4)


In [ ]:
print(path)
os.listdir(path)

/root/.cache/kagglehub/datasets/aryayadav0513/m5-forecasting-accuracy/versions/1


['m5-forecasting-accuracy']

In [ ]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

In [ ]:

sales = sales[
    (sales['state_id'].isin(['CA','TX','WI'])) &
    (sales['cat_id'].isin(['HOBBIES','HOUSEHOLD','FOODS']))
]

sales = sales.sample(frac=0.5, random_state=42)


day_cols = [c for c in sales.columns if 'd_' in c]
half_days = day_cols[:len(day_cols)//2]

sales = sales[['item_id','dept_id','cat_id','store_id','state_id'] + half_days]

In [ ]:
sales_long = sales.melt(
    id_vars=['item_id','dept_id','cat_id','store_id','state_id'],
    var_name='d',
    value_name='sales'
)

In [ ]:
sales_long = sales_long.merge(
    calendar[['d','wm_yr_wk']],
    on='d',
    how='left'
)

In [ ]:
sales_long = sales_long.merge(
    prices,
    on=['store_id','item_id','wm_yr_wk'],
    how='left'
)

### Feature Engineering: Adding Temporal Features

To provide the LSTM model with more context and potentially improve its performance, we will add temporal features such as `wday` (day of the week) and `month` to our `weekly_avg` dataset. These features can help the model identify recurring patterns related to specific days or months.

In [ ]:
# Merge weekly_avg with calendar to add wday and month information
# First, create a mapping from wm_yr_wk to a representative date, wday, and month
calendar_features = calendar[['wm_yr_wk', 'wday', 'month']].drop_duplicates('wm_yr_wk')

# Ensure wm_yr_wk is the index to easily identify missing ones
weekly_avg_enriched = weekly_avg.merge(calendar_features, on='wm_yr_wk', how='left')

print("weekly_avg with new features:")
display(weekly_avg_enriched.head())

# Check for any missing values after merge (should ideally be none if calendar covers all wm_yr_wk)
print("Missing values after merging calendar features:")
print(weekly_avg_enriched.isnull().sum())

NameError: name 'weekly_avg' is not defined

## Model Optimization using Keras Tuner

To improve the performance of our LSTM model, we will perform hyperparameter tuning using `keras-tuner`. This process involves searching for the best combination of parameters (like the number of LSTM units, layers, and learning rate) that yield the best performance on a validation set.

First, we need to install the `keras-tuner` library.

In [ ]:
# Install Keras Tuner
!pip install keras-tuner -q

Next, we define a `build_model` function that will create our LSTM model with tunable hyperparameters. We'll make the number of LSTM units, the number of LSTM layers, and the learning rate for the Adam optimizer tunable.

In [ ]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.models import Model # Import Model for Functional API
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input # Import Input

def build_model(hp):
    # Define input layer using Functional API
    inputs = Input(shape=(10, 3)) # Explicitly define input shape (time_steps, num_features)
    x = inputs

    # Tune the number of Conv1D layers
    num_cnn_layers = hp.Int('num_cnn_layers', min_value=1, max_value=2, step=1)

    for i in range(num_cnn_layers):
        # Tune the number of filters and kernel size for each layer
        num_filters = hp.Int(f'filters_{i}', min_value=32, max_value=128, step=32)
        kernel_size = hp.Choice(f'kernel_size_{i}', values=[2, 3, 5])

        x = Conv1D(filters=num_filters, kernel_size=kernel_size, activation='relu')(x)
        # Removed MaxPooling1D from here to avoid reducing sequence length too quickly

    x = Flatten()(x)
    outputs = Dense(1)(x)

    model = Model(inputs=inputs, outputs=outputs) # Create model using Functional API

    # Tune the learning rate for the Adam optimizer
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(loss='mse', optimizer=optimizer)
    return model

Now, we'll perform the hyperparameter search using `RandomSearch`. To keep the tuning process manageable, we'll run it on a representative sample of data, for example, the 'HOBBIES' category in 'CA' state. Once the best hyperparameters are found, we will use them to train our LSTM models across all categories and states.

Now we will modify the `train_lstm` function to use these `best_hps` for training, thereby optimizing our model. We will then rerun the plotting and evaluation cells to see the impact of these optimized hyperparameters.

In [ ]:
hobbies = weekly_avg_enriched[weekly_avg_enriched['cat_id']=='HOBBIES']
household = weekly_avg_enriched[weekly_avg_enriched['cat_id']=='HOUSEHOLD']
foods = weekly_avg_enriched[weekly_avg_enriched['cat_id']=='FOODS']

In [ ]:
def create_dataset(data, time_step=10, num_features=1):
    X, Y = [], []
    for i in range(len(data)-time_step-1):
        X.append(data[i:(i+time_step), :num_features]) # Take all features for X
        Y.append(data[i+time_step, 0]) # Assuming 'revenue' (first feature) is the target
    return np.array(X), np.array(Y)

In [ ]:
def wrmsse_score(y_true, y_pred):
    """
    Calculates a simplified WRMSSE for a single time series,
    where the scaling factor is derived from the historical actuals (`y_true`).

    y_true: Actual values corresponding to the predictions.
    y_pred: Predicted values.
    """
    rmse = np.sqrt(np.mean((y_pred - y_true)**2))

    if len(y_true) > 1:
        # M5-like scaling: RMS of first differences
        # Using np.diff(y_true) calculates y_true[i] - y_true[i-1]
        diffs = np.diff(y_true)
        if len(diffs) > 0:
            scale = np.sqrt(np.mean(diffs**2))
            if scale == 0:
                # If historical data has no variance, WRMSSE is RMSE
                return rmse
            return rmse / scale
        else:
            # Not enough data for diffs, return RMSE
            return rmse
    else:
        # Not enough data for scaling, return RMSE
        return rmse

In [ ]:
# Select a representative series for tuning (e.g., HOBBIES CA)
# This is done to limit computation time for hyperparameter search
# Fix: Ensure sample_series contains 'wday' and 'month' by deriving it from weekly_avg_enriched
sample_series = weekly_avg_enriched[
    (weekly_avg_enriched['cat_id']=='HOBBIES') &
    (weekly_avg_enriched['state_id']=='CA')
].sort_values('wm_yr_wk')

# Prepare features for tuning
# For CNN, we need multiple features as input. Let's use revenue, wday, month.
# The create_dataset function will handle this.
combined_data_for_tune = sample_series[['revenue', 'wday', 'month']].values

# Scale the sample series
scaler_tune = MinMaxScaler()
combined_scaled_data_for_tune = scaler_tune.fit_transform(combined_data_for_tune)

# Create dataset for the tuner
# num_features=3 for revenue, wday, month
X_tune, y_tune = create_dataset(combined_scaled_data_for_tune, time_step=10, num_features=3)
X_tune = X_tune.reshape(X_tune.shape[0], X_tune.shape[1], 3) # Reshape for Conv1D input (samples, timesteps, features)

# Instantiate the Keras Tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,  # Number of hyperparameter combinations to try
    executions_per_trial=1, # Number of models to train for each trial (for robustness)
    directory='keras_tuner_dir', # Directory to store results
    project_name='cnn_tuning_v3', # Changed project name to force a new tuner instance and clear cache
    overwrite=True # Overwrite existing project data
)

print("Starting hyperparameter search...")
# Run the hyperparameter search
tuner.search(
    X_tune, y_tune,
    epochs=3, # Fewer epochs for quick tuning, full training will use more
    batch_size=32,
    validation_split=0.2,
    verbose=0
)
print("Hyperparameter search complete.")

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"\nOptimal Number of CNN Layers: {best_hps.get('num_cnn_layers')}")
for i in range(best_hps.get('num_cnn_layers')):
    print(f"Optimal Filters (Layer {i}): {best_hps.get(f'filters_{i}')}")
    print(f"Optimal Kernel Size (Layer {i}): {best_hps.get(f'kernel_size_{i}')}")
print(f"Optimal Learning Rate: {best_hps.get('learning_rate')}")

In [ ]:
import lightgbm as lgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Assuming create_dataset and wrmsse_score are defined in an earlier cell.

def train_lgbm(df_series):
    """
    Trains a LightGBM model on the given time series data.

    Args:
        df_series (pd.DataFrame): Input DataFrame with 'revenue', 'month', 'wday' columns.

    Returns:
        tuple: (pred, real, mae, rmse, mape, wrmsse) - predictions, actuals, and evaluation metrics.
    """
    features_to_use = ['revenue', 'wday', 'month']
    features_data = df_series[features_to_use].values

    # Scale the features
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(features_data)

    num_features = scaled_data.shape[1]
    time_step = 10 # This should match the time_step used in create_dataset

    X, y = create_dataset(scaled_data, time_step=time_step)

    # Split data into training and testing sets
    split = int(len(X) * 0.8)
    X_train = X[:split]
    X_test = X[split:]
    y_train = y[:split]
    y_test = y[split:]

    # Initialize and train LightGBM Regressor
    # Using default hyperparameters for now, tuning can be added later if needed.
    lgbm_model = lgb.LGBMRegressor(random_state=42)
    lgbm_model.fit(X_train, y_train)

    # Make predictions
    pred_scaled = lgbm_model.predict(X_test)

    # Inverse transform predictions and actuals
    # Predictions are for 'revenue' (first feature)
    dummy_pred_array = np.zeros((len(pred_scaled), num_features))
    dummy_pred_array[:, 0] = pred_scaled
    pred = scaler.inverse_transform(dummy_pred_array)[:, 0]

    dummy_real_array = np.zeros((len(y_test), num_features))
    dummy_real_array[:, 0] = y_test
    real = scaler.inverse_transform(dummy_real_array)[:, 0]

    # Calculate evaluation metrics
    mae = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    # MAPE calculation: handle division by zero or very small real values
    mape = np.mean(np.abs((real - pred) / (real + 1e-8))) * 100
    wrmsse = wrmsse_score(real.flatten(), pred.flatten())

    return pred, real, mae, rmse, mape, wrmsse

In [ ]:
all_metrics = []

def plot_category(data, title):

    plt.figure(figsize=(12,5))

    for state in ['CA','TX','WI']:

        df = data[
            data['state_id']==state
        ].sort_values('wm_yr_wk')

        if len(df) < 30:
            continue

        # Changed to call train_lgbm
        pred, real, mae, rmse, mape, rmsse = train_lgbm(df)

        all_metrics.append({
            'Category': title,
            'State': state,
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape,
            'RMSSE': rmsse
        })

        print(f"\n{title} - {state}")
        print(f"MAE   : {mae:.2f}")
        print(f"RMSE  : {rmse:.2f}")
        print(f"MAPE  : {mape:.2f}%")
        print(f"RMSSE : {rmsse:.4f}")

        plt.plot(
            real,
            label=f"{state} Actual"
        )

        plt.plot(
            pred,
            '--',
            label=f"{state} LightGBM" # Changed label
        )

    plt.title(title)

    plt.xlabel("Week")

    plt.ylabel("Revenue")

    plt.legend()

    plt.show()

In [ ]:
plot_category(hobbies, "HOBBIES Revenue Forecast")

In [ ]:
plot_category(household, "HOUSEHOLD Revenue Forecast")

In [ ]:
plot_category(foods, "FOODS Revenue Forecast")

In [ ]:
import tensorflow as tf
import numpy as np
import random
import pandas as pd # Ensure pandas is imported for DataFrame

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

all_metrics = [] # Moved initialization here to ensure it's cleared before population

# These calls will now use the modified plot_category which calls train_lgbm
plot_category(hobbies, "HOBBIES Revenue Forecast")
plot_category(household, "HOUSEHOLD Revenue Forecast")
plot_category(foods, "FOODS Revenue Forecast")

df_res = pd.DataFrame(all_metrics)

overall_mae = df_res['MAE'].mean()
overall_rmse = df_res['RMSE'].mean()
overall_mape = df_res['MAPE'].mean()
overall_wrmsse = df_res['RMSSE'].mean() # Using RMSSE from the dictionary, which is actually WRMSSE from the function.

overall_accuracy = 100 - overall_mape

tabel_evaluasi = pd.DataFrame({
    'Model': ['LightGBM'], # Updated model name from 'CNN' to 'LightGBM'
    'MAE': [round(overall_mae, 4)],
    'RMSE': [round(overall_rmse, 4)],
    'MAPE': [f"{overall_mape:.2f}%"],
    'WRMSSE': [f"{overall_wrmsse:.4f}"],
    'Akurasi': [f"{overall_accuracy:.2f}%"],
})

print("\nTABEL 1. Evaluasi Kinerja Model Deep Learning")
display(tabel_evaluasi)

### Note on Evaluation Metrics

It's important to differentiate between the metrics presented in **TABEL 1** (MAE, RMSE, MAPE for the LSTM model) and the **WRMSSE** calculated later in the notebook.

*   **TABEL 1** evaluates the LSTM model's performance on **aggregated weekly revenue series** for specific categories and states. These are standard regression metrics applied to time series data.

*   The **WRMSSE (Weighted Root Mean Squared Scaled Error)**, as used in the M5 competition, is a more complex metric applied at the **individual item-store level**. The LSTM model in this notebook is not designed to forecast at this granular level, and therefore, its performance is not directly measured by the M5 WRMSSE. The WRMSSE calculation provided in the subsequent cells serves as an illustration of how this metric is computed, using a **naive forecast** as an example, rather than evaluating the current LSTM model's WRMSSE.

### Weighted Root Mean Squared Scaled Error (WRMSSE) Results

Below are the WRMSSE results, calculated using a naive forecast (last 28 days of training data as prediction). This metric accounts for both the scale of sales and the financial importance of each item.

In [ ]:
# This cell is now empty as its content has been moved to caddb439

### Overall Evaluation Metrics Visualization

Let's visualize the Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Mean Absolute Percentage Error (MAPE) for each category and state to get an overall understanding of the model's performance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Added to ensure DataFrame creation

# Ensure df_res is defined. This assumes all_metrics has been populated by the previous cell (_3eO8G6Fa1Bc).
# If 'all_metrics' is not defined or is empty, this may still lead to issues, but it resolves the NameError for 'df_res'.
df_res = pd.DataFrame(all_metrics)

# Visualize MAE across categories and states
plt.figure(figsize=(12, 6))
sns.barplot(data=df_res, x='Category', y='MAE', hue='State', palette='viridis')
plt.title('Mean Absolute Error (MAE) by Category and State')
plt.xlabel('Category')
plt.ylabel('MAE')
plt.xticks(rotation=45)
plt.legend(title='State')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Added to ensure DataFrame creation

# Ensure df_res is defined. This assumes all_metrics has been populated by the previous cell (_3eO8G6Fa1Bc).
# If 'all_metrics' is not defined or is empty, this may still lead to issues, but it resolves the NameError for 'df_res'.
df_res = pd.DataFrame(all_metrics)

# Visualize RMSE across categories and states
plt.figure(figsize=(12, 6))
sns.barplot(data=df_res, x='Category', y='RMSE', hue='State', palette='magma')
plt.title('Root Mean Squared Error (RMSE) by Category and State')
plt.xlabel('Category')
plt.ylabel('RMSE')
plt.xticks(rotation=45)
plt.legend(title='State')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Added to ensure DataFrame creation

# Ensure df_res is defined. This assumes all_metrics has been populated by the previous cell (_3eO8G6Fa1Bc).
# If 'all_metrics' is not defined or is empty, this may still lead to issues, but it resolves the NameError for 'df_res'.
df_res = pd.DataFrame(all_metrics)

# Visualize MAPE across categories and states
plt.figure(figsize=(12, 6))
sns.barplot(data=df_res, x='Category', y='MAPE', hue='State', palette='plasma')
plt.title('Mean Absolute Percentage Error (MAPE) by Category and State')
plt.xlabel('Category')
plt.ylabel('MAPE (%)')
plt.xticks(rotation=45)
plt.legend(title='State')
plt.tight_layout()
plt.show()

### Calculate Weighted Root Mean Squared Scaled Error (WRMSSE)

The WRMSSE is the evaluation metric used in the M5 Forecasting Accuracy competition. It's a weighted average of the Root Mean Squared Scaled Error (RMSSE) across all items and stores. The calculation requires historical sales data from both `sales_train_validation.csv` and `sales_train_evaluation.csv`, as well as `sell_prices.csv` to determine the weights.

Here's a breakdown of the steps:
1.  **Load Sales Evaluation Data**: Load `sales_train_evaluation.csv` to get the actual sales for the evaluation period.
2.  **Calculate Denominator for RMSSE**: This involves the daily differences of the training sales data.
3.  **Calculate Weights**: Weights are based on the sales revenue (sell price * sales) during the validation period.
4.  **Calculate RMSSE**: For each item, calculate the Root Mean Squared Scaled Error.
5.  **Calculate WRMSSE**: Sum the weighted RMSSE values.

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

print('Loading sales_train_evaluation.csv...')
sales_wrmsse = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv")) # Load full sales_train_validation.csv
sales_eval_wrmsse = pd.read_csv(os.path.join(dataset_path, "sales_train_evaluation.csv"))
print('Done.')

In [ ]:
print('Preparing sales data for WRMSSE calculation...')

# Debug prints BEFORE strip and filter
print("\n--- Debug: sales_wrmsse BEFORE strip and filter ---")
print("Shape:", sales_wrmsse.shape)
print("Unique states:", sales_wrmsse['state_id'].unique())
print("Unique categories:", sales_wrmsse['cat_id'].unique())

print("\n--- Debug: sales_eval_wrmsse BEFORE strip and filter ---")
print("Shape:", sales_eval_wrmsse.shape)
print("Unique states:", sales_eval_wrmsse['state_id'].unique())
print("Unique categories:", sales_eval_wrmsse['cat_id'].unique())

# Clean 'state_id' and 'cat_id' columns by stripping whitespace
sales_eval_wrmsse['state_id'] = sales_eval_wrmsse['state_id'].str.strip()
sales_eval_wrmsse['cat_id'] = sales_eval_wrmsse['cat_id'].str.strip()
sales_wrmsse['state_id'] = sales_wrmsse['state_id'].str.strip()
sales_wrmsse['cat_id'] = sales_wrmsse['cat_id'].str.strip()

# Debug prints AFTER strip
print("\n--- Debug: sales_wrmsse AFTER strip ---")
print("Unique states:", sales_wrmsse['state_id'].unique())
print("Unique categories:", sales_wrmsse['cat_id'].unique())

print("\n--- Debug: sales_eval_wrmsse AFTER strip ---")
print("Unique states:", sales_eval_wrmsse['state_id'].unique())
print("Unique categories:", sales_eval_wrmsse['cat_id'].unique())

# Filter sales_eval_wrmsse similar to how sales was filtered in earlier steps for consistency
sales_eval_wrmsse = sales_eval_wrmsse[
    (sales_eval_wrmsse['state_id'].isin(['CA','TX','WI'])) &
    (sales_eval_wrmsse['cat_id'].isin(['HOBBIES','HOUSEHOLD','FOODS']))
]

sales_wrmsse = sales_wrmsse[
    (sales_wrmsse['state_id'].isin(['CA','TX','WI'])) &
    (sales_wrmsse['cat_id'].isin(['HOBBIES','HOUSEHOLD','FOODS']))
]

# Debug prints AFTER filter
print("\n--- Debug: sales_wrmsse AFTER filter ---")
print("Shape:", sales_wrmsse.shape)

print("\n--- Debug: sales_eval_wrmsse AFTER filter ---")
print("Shape:", sales_eval_wrmsse.shape)

# Columns that uniquely identify a sales series (excluding the 'id' which has _validation/_evaluation suffix)
series_id_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

# Get the set of unique series identifiers from both filtered dataframes
sales_wrmsse_unique_series = sales_wrmsse[series_id_cols].drop_duplicates()
sales_eval_wrmsse_unique_series = sales_eval_wrmsse[series_id_cols].drop_duplicates()

# Find the common series (item/store/category/state combinations) that exist in both
common_series_identifiers = pd.merge(sales_wrmsse_unique_series,
                                     sales_eval_wrmsse_unique_series,
                                     on=series_id_cols,
                                     how='inner')

# Filter sales_wrmsse and sales_eval_wrmsse to keep only these common series
sales_wrmsse = pd.merge(common_series_identifiers, sales_wrmsse, on=series_id_cols, how='left')
sales_eval_wrmsse = pd.merge(common_series_identifiers, sales_eval_wrmsse, on=series_id_cols, how='left')

# Sort both dataframes by their respective 'id' columns to ensure row alignment
# (sales_wrmsse 'id' ends in _validation, sales_eval_wrmsse 'id' ends in _evaluation)
# This will ensure that corresponding rows represent the same item series
sales_wrmsse = sales_wrmsse.sort_values(by='id').reset_index(drop=True)
sales_eval_wrmsse = sales_eval_wrmsse.sort_values(by='id').reset_index(drop=True)


# Align columns - sales_wrmsse has validation days, sales_eval_wrmsse has all training + evaluation days
d_cols_validation_wrmsse = [c for c in sales_wrmsse.columns if 'd_' in c]
d_cols_evaluation_wrmsse = [c for c in sales_eval_wrmsse.columns if 'd_' in c]

# Fill any potential NaN from merging with 0 sales for days not present in original `sales` after filtering
sales_wrmsse = sales_wrmsse.fillna(0)
sales_eval_wrmsse = sales_eval_wrmsse.fillna(0)

# Define 'common_cols' to be used in subsequent cells for selecting non-day columns
common_cols = ['id'] + series_id_cols # This 'common_cols' refers to all identifying columns including 'id'


print('\nDone.')

In [ ]:
# Calculate the denominator for RMSSE (scale)
# s_i,j = sqrt(1/(n-1) * sum((y_t - y_bar)^2))

print('Calculating RMSSE denominator...')

d_cols_sales = [col for col in sales_wrmsse.columns if 'd_' in col] # Use sales_wrmsse

diff = sales_wrmsse[d_cols_sales].diff(axis=1)
sq_diff = np.square(diff.iloc[:, 1:]) # Skip the first column as diff will be NaN

n_minus_1 = len(d_cols_sales) - 1

denominator = np.sqrt(sq_diff.sum(axis=1) / n_minus_1)

# Handle cases where denominator might be zero (e.g., constant sales)
denominator[denominator == 0] = 1 # Avoid division by zero, set to 1 for items with no variance

sales_wrmsse['denominator'] = denominator # Add denominator to sales_wrmsse
print('Done.')

In [ ]:
print('Calculating item weights...')

# Weights are based on the sales revenue during the validation period (last 28 days of training)
# The validation period for M5 is d_1886 to d_1913

# Get the last 28 days of sales data from the 'sales_wrmsse' dataframe (which is sales_train_validation)
# Assuming d_cols_validation_wrmsse contains all 'd_' columns up to d_1913
validation_start_day = max([int(c.replace('d_', '')) for c in d_cols_validation_wrmsse]) - 27 # d_1886
validation_cols = [f'd_{i}' for i in range(validation_start_day, max([int(c.replace('d_', '')) for c in d_cols_validation_wrmsse]) + 1)]

sales_validation_period = sales_wrmsse[validation_cols]

# Melt sales_validation_period to join with prices
sales_validation_long = sales_wrmsse[common_cols].copy()
sales_validation_long = pd.melt(
sales_validation_long.assign(**sales_validation_period),
    id_vars=common_cols,
    var_name='d',
    value_name='sales'
)

# Merge with calendar to get wm_yr_wk for prices
sales_validation_long = sales_validation_long.merge(
    calendar[['d', 'wm_yr_wk']],
    on='d',
    how='left'
)

# Merge with prices to get sell_price
sales_validation_long = sales_validation_long.merge(
    prices,
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)

# Calculate revenue during validation period
sales_validation_long['revenue'] = sales_validation_long['sales'] * sales_validation_long['sell_price']

# Sum revenue by item_id to get weights
weights_df = sales_validation_long.groupby(['item_id', 'store_id', 'dept_id', 'cat_id', 'state_id'])['revenue'].sum().reset_index()
weights_df.rename(columns={'revenue': 'weight'}, inplace=True)

# Normalize weights
weights_df['weight'] = weights_df['weight'] / weights_df['weight'].sum()

# Merge weights back to sales_eval_wrmsse for WRMSSE calculation
sales_eval_wrmsse = sales_eval_wrmsse.merge(weights_df, on=['item_id', 'store_id', 'dept_id', 'cat_id', 'state_id'], how='left')
sales_eval_wrmsse['weight'] = sales_eval_wrmsse['weight'].fillna(0) # Fill NaN weights with 0

del sales_validation_long
del weights_df
gc.collect()

print('Done.')

In [ ]:
print('Calculating WRMSSE...')

# Define the forecast horizon (e.g., last 28 days of sales_eval for M5)
# This assumes the model has predicted these days, but for WRMSSE we need actuals

# For this demonstration, let's assume `y` from `train_lstm` is our 'forecast'
# and `real` is our 'actuals' for the last 28 days of `sales_eval_wrmsse`.
# However, the previous LSTM model is trained on a weekly aggregation and a sample of data.
# To correctly calculate WRMSSE, we need to apply the forecasting model to each individual series
# in `sales_eval_wrmsse` and then compare.

# Given the complexity and time for training for each item/store combination from scratch for a full WRMSSE,
# and that the previous LSTM was trained on *aggregated* weekly data,
# we will demonstrate the WRMSSE calculation structure using the individual series and
# a simplified 'forecast' based on a naive approach for illustration purposes.
# A full WRMSSE would involve retraining or applying a model to each series.

# Let's use the `sales_eval_wrmsse` data to represent the actuals (y_true) and a simple forecast (y_pred)
# For simplicity, let's assume our 'forecast' for the evaluation period (d_1914 to d_1941) is
# the last 28 days of the *training* data (d_1886 to d_1913). This is a common naive forecast.

# Get evaluation period columns (d_1914 to d_1941) - using d_cols_evaluation_wrmsse
eval_start_day = max([int(c.replace('d_', '')) for c in d_cols_evaluation_wrmsse]) - 27 # d_1914
prediction_cols = [f'd_{i}' for i in range(eval_start_day, max([int(c.replace('d_', '')) for c in d_cols_evaluation_wrmsse]) + 1)]

# Actuals for the evaluation period
y_true = sales_eval_wrmsse[prediction_cols].values

# Naive forecast: last 28 days of training data (from `sales_wrmsse` dataframe)
# This is d_1886 to d_1913 from the sales_train_validation.csv
# This assumes sales_wrmsse has been filtered and processed as needed for alignment.

# Ensure `sales_wrmsse` has the necessary columns for the naive forecast
# This implies that `sales_wrmsse` contains d_1886 to d_1913
forecast_cols_for_naive = [f'd_{i}' for i in range(validation_start_day, validation_start_day + 28)] # d_1886 to d_1913

y_pred = sales_wrmsse[forecast_cols_for_naive].values


# Calculate squared errors for the forecast period
sq_err = np.square(y_true - y_pred)

# Sum of squared errors for each item (across the forecast horizon)
sum_sq_err = np.sum(sq_err, axis=1)

# Mean Squared Error for each item
mse_per_item = sum_sq_err / len(prediction_cols)

# Root Mean Squared Error for each item
rmse_per_item = np.sqrt(mse_per_item)

# Calculate RMSSE for each item (using the pre-calculated denominator)
# We need to align rmse_per_item with the correct denominator.
# Add rmse_per_item to sales_eval_wrmsse temporarily for alignment with denominator
sales_eval_wrmsse['rmse_per_item'] = rmse_per_item

rmsse_per_item = sales_eval_wrmsse['rmse_per_item'] / sales_wrmsse['denominator'] # Use sales_wrmsse for denominator

# Calculate Weighted RMSSE (WRMSSE)
wrmsse = np.sum(rmsse_per_item * sales_eval_wrmsse['weight'])

print(f"Calculated overall WRMSSE: {wrmsse:.10f}")

# Calculate WRMSSE per state
sales_eval_wrmsse['rmsse_per_item'] = rmsse_per_item # Ensure this is updated if not already

wrmsse_per_state = sales_eval_wrmsse.groupby('state_id').apply(lambda x: np.sum(x['rmsse_per_item'] * x['weight']), include_groups=False)

print("\nCalculated WRMSSE per state:")
for state, val in wrmsse_per_state.items():
    print(f"  {state}: {val:.10f}")

print('\nDone.')

### WRMSSE Result

The Weighted Root Mean Squared Scaled Error (WRMSSE) for the predictions (using a naive forecast as an example) is presented below. This metric accounts for both the scale of sales and the financial importance of each item.

**Note**: The WRMSSE calculated here is based on a simplified naive forecast (last 28 days of training as prediction for evaluation). For a true WRMSSE, the actual predictions from your LSTM model (or any other forecasting model) for the evaluation period would be used.

## Bagian Tambahan: Perbandingan Adil (Walk-Forward) 1D-CNN vs ARIMA + Log Hyperparameter Tuning

Bagian ini menambahkan skema evaluasi **walk-forward** yang **identik** untuk 1D-CNN dan ARIMA supaya perbandingannya adil, sekaligus mencatat semua hasil **hyperparameter tuning** dari kedua model (bukan cuma nilai terbaiknya, tapi seluruh kombinasi yang dicoba) supaya bisa langsung dikutip/dilampirkan di laporan riset.

**Protokol yang dipakai (sama untuk kedua model):**
1. Setiap deret mingguan (kombinasi *state* × *kategori*) dibagi: `TEST_WEEKS` minggu terakhir (default 30, otomatis menyesuaikan jika data lebih pendek) jadi periode uji *walk-forward*; sisanya jadi data latih awal.
2. Di setiap langkah minggu uji ke-*t*:
   - **ARIMA**: di-*fit ulang* memakai seluruh histori yang tersedia sampai minggu *t-1* (expanding window), dengan orde (p,d,q) yang sudah dipilih lewat grid search di data latih awal, lalu forecast 1 langkah ke depan (h=1).
   - **1D-CNN**: juga di-*retrain ulang* di setiap langkah memakai seluruh histori yang tersedia sampai *t-1* (expanding window, sama seperti ARIMA), dengan arsitektur & hyperparameter tetap (hasil tuning Keras Tuner), input berupa jendela `TIME_STEP` minggu terakhir, lalu memprediksi 1 minggu ke depan.
   - Kedua model memprediksi **minggu target yang sama persis**, dari data historis yang tersedia sama persis di setiap langkah → hasilnya benar-benar apple-to-apple.
3. Skala penyebut WRMSSE (RMS dari selisih pertama/naive) dihitung dari **10 minggu terakhir sebelum periode uji dimulai** — sama persis untuk kedua model, sesuai deskripsi "10-week historical lookback truncation" di metodologi ARIMA pada paper.
4. Hyperparameter CNN di-tuning **satu kali** di deret representatif (HOBBIES-CA) dan dipakai untuk semua kombinasi state/kategori — ini konsisten dengan Tabel 2 di paper (satu set hyperparameter per model). Orde ARIMA di-tuning **per deret** (per kombinasi state × kategori) karena karakteristik ARIMA memang harus disesuaikan per deret waktu — ini juga konsisten dengan deskripsi metodologi ARIMA di paper ("parameter order tetap berdasarkan initial training data").

> Catatan performa: karena CNN di-retrain di setiap langkah walk-forward (bukan cuma sekali), proses ini lebih lambat dari training CNN biasa. Kalau mau lebih cepat saat mencoba-coba, kecilkan `TEST_WEEKS` dan/atau `CNN_EPOCHS` di sel konfigurasi di bawah.


In [ ]:
# ==========================================================
# Rebuild weekly_avg_enriched (self-contained, robust)
# Menggunakan sales_long (item_id, dept_id, cat_id, store_id,
# state_id, d, sales, wm_yr_wk, sell_price) + calendar
# Kalau weekly_avg_enriched sudah ada dan valid, tidak perlu dibangun ulang.
# ==========================================================
import numpy as np
import pandas as pd

need_rebuild = True
try:
    need_rebuild = ('weekly_avg_enriched' not in globals()) or \
                   (not {'revenue','wday','month','cat_id','state_id','wm_yr_wk'}.issubset(weekly_avg_enriched.columns))
except NameError:
    need_rebuild = True

if need_rebuild:
    print("Membangun ulang weekly_avg_enriched dari sales_long + calendar ...")
    sl = sales_long.copy()
    sl['sales'] = pd.to_numeric(sl['sales'], errors='coerce').fillna(0)
    sl['sell_price'] = pd.to_numeric(sl['sell_price'], errors='coerce')
    sl['revenue_row'] = sl['sales'] * sl['sell_price'].fillna(0)

    weekly_avg = (
        sl.groupby(['wm_yr_wk', 'state_id', 'cat_id'], as_index=False)
          .agg(revenue=('revenue_row', 'sum'))
    )

    calendar_features = calendar[['wm_yr_wk', 'wday', 'month']].drop_duplicates('wm_yr_wk')
    weekly_avg_enriched = weekly_avg.merge(calendar_features, on='wm_yr_wk', how='left')
    weekly_avg_enriched = weekly_avg_enriched.sort_values(['state_id', 'cat_id', 'wm_yr_wk']).reset_index(drop=True)

    print("weekly_avg_enriched shape:", weekly_avg_enriched.shape)
    display(weekly_avg_enriched.head())
else:
    print("weekly_avg_enriched sudah tersedia, memakai yang sudah ada.")
    display(weekly_avg_enriched.head())


In [ ]:
# ==========================================================
# Konfigurasi Walk-Forward (SAMA untuk 1D-CNN dan ARIMA)
# ==========================================================
import warnings
warnings.filterwarnings("ignore")

TIME_STEP   = 10   # panjang jendela input (minggu) -> sama dengan Tabel 2 paper (Time Step = 10)
TEST_WEEKS  = 30   # panjang periode uji walk-forward (minggu) -> sama dengan metodologi ARIMA di paper
NUM_FEATURES = 3   # revenue, wday, month
CNN_EPOCHS  = 15   # epoch training CNN di SETIAP langkah walk-forward (retrain ulang tiap minggu)
CNN_BATCH   = 8

CATEGORIES = ['HOBBIES', 'HOUSEHOLD', 'FOODS']
STATES     = ['CA', 'TX', 'WI']

print(f"TIME_STEP={TIME_STEP}, TEST_WEEKS={TEST_WEEKS}, CNN_EPOCHS={CNN_EPOCHS}, CNN_BATCH={CNN_BATCH}")


### 1. Hyperparameter Tuning — 1D-CNN (Keras Tuner)

Tuning dilakukan pada deret representatif (HOBBIES–CA), memakai `RandomSearch` dari Keras Tuner. **Semua trial (bukan cuma yang terbaik) dicatat** di tabel `cnn_tuning_results` supaya bisa langsung dilampirkan sebagai bukti proses tuning di laporan riset.


In [ ]:
# ==========================================================
# 1D-CNN Hyperparameter Tuning (Keras Tuner - RandomSearch)
# ==========================================================
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input
from sklearn.preprocessing import MinMaxScaler

tf.random.set_seed(42)
np.random.seed(42)

def create_dataset(data, time_step=10, num_features=1):
    """Sliding-window supervised dataset. Target = kolom ke-0 (revenue)."""
    X, Y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:(i + time_step), :num_features])
        Y.append(data[i + time_step, 0])
    return np.array(X), np.array(Y)

def build_model(hp):
    inputs = Input(shape=(TIME_STEP, NUM_FEATURES))
    x = inputs
    num_cnn_layers = hp.Int('num_cnn_layers', min_value=1, max_value=3, step=1)
    for i in range(num_cnn_layers):
        num_filters = hp.Int(f'filters_{i}', min_value=32, max_value=128, step=32)
        kernel_size = hp.Choice(f'kernel_size_{i}', values=[2, 3, 5])
        x = Conv1D(filters=num_filters, kernel_size=kernel_size,
                   activation='relu', padding='causal')(x)
    x = Flatten()(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate))
    return model

# --- data tuning: deret representatif HOBBIES-CA ---
tuning_series = weekly_avg_enriched[
    (weekly_avg_enriched['cat_id'] == 'HOBBIES') &
    (weekly_avg_enriched['state_id'] == 'CA')
].sort_values('wm_yr_wk')

feat_tune = tuning_series[['revenue', 'wday', 'month']].values
scaler_tune = MinMaxScaler()
feat_tune_scaled = scaler_tune.fit_transform(feat_tune)
X_tune, y_tune = create_dataset(feat_tune_scaled, TIME_STEP, NUM_FEATURES)
print("Bentuk data tuning:", X_tune.shape)

tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=15,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='cnn_walkforward_tuning',
    overwrite=True,
)

print("Menjalankan hyperparameter search untuk 1D-CNN...")
tuner.search(X_tune, y_tune, epochs=15, batch_size=16, validation_split=0.2, verbose=0)
print("Selesai.")

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_hp_values = dict(best_hps.values)
print("\n=== Hyperparameter 1D-CNN terbaik ===")
for k, v in best_hp_values.items():
    print(f"{k}: {v}")

# --- catat SEMUA trial (untuk dilampirkan di laporan) ---
trial_records = []
for trial_id, trial in tuner.oracle.trials.items():
    rec = dict(trial.hyperparameters.values)
    rec['trial_id'] = trial_id
    rec['val_loss'] = trial.score
    trial_records.append(rec)

cnn_tuning_results = pd.DataFrame(trial_records).sort_values('val_loss').reset_index(drop=True)
cnn_tuning_results.insert(0, 'rank', range(1, len(cnn_tuning_results) + 1))

print("\n=== Log Lengkap Hyperparameter Tuning 1D-CNN (semua trial) ===")
display(cnn_tuning_results)

cnn_tuning_results.to_csv('cnn_hyperparameter_tuning_log.csv', index=False)
print("\nDisimpan ke: cnn_hyperparameter_tuning_log.csv")


### 2. Hyperparameter Tuning — ARIMA (Grid Search Orde p, d, q)

Untuk ARIMA, "hyperparameter" yang di-tuning adalah orde `(p, d, q)`. Sesuai batasan di metodologi paper (`p ≤ 2, d ≤ 1, q ≤ 2`), dilakukan **grid search berbasis AIC** — dan dilakukan **per deret** (per kombinasi *state* × *kategori*) karena karakteristik tiap deret berbeda. Semua kombinasi yang dicoba, beserta AIC-nya, dicatat di `arima_tuning_results_all` supaya bisa dilampirkan di laporan, sedangkan orde terbaik per deret ada di `arima_best_orders`.


In [ ]:
# ==========================================================
# ARIMA Hyperparameter Tuning (Grid Search p<=2, d<=1, q<=2, per deret)
# ==========================================================
import itertools
from statsmodels.tsa.arima.model import ARIMA

def arima_order_search(train_series, p_max=2, d_max=1, q_max=2):
    """Grid search orde ARIMA berbasis AIC pada data latih awal (sebelum periode uji walk-forward)."""
    best_aic = np.inf
    best_order = (0, 1, 0)
    records = []
    for p, d, q in itertools.product(range(p_max + 1), range(d_max + 1), range(q_max + 1)):
        try:
            fit = ARIMA(train_series, order=(p, d, q)).fit()
            records.append({'p': p, 'd': d, 'q': q, 'AIC': fit.aic})
            if fit.aic < best_aic:
                best_aic = fit.aic
                best_order = (p, d, q)
        except Exception:
            continue
    return best_order, best_aic, pd.DataFrame(records)

arima_best_orders = []
arima_all_trials = []

for cat in CATEGORIES:
    for state in STATES:
        series_df = weekly_avg_enriched[
            (weekly_avg_enriched['cat_id'] == cat) &
            (weekly_avg_enriched['state_id'] == state)
        ].sort_values('wm_yr_wk')

        if len(series_df) < TIME_STEP + 20:
            print(f"Lewati {cat}-{state}: data terlalu pendek ({len(series_df)} minggu)")
            continue

        n_test = min(TEST_WEEKS, len(series_df) - TIME_STEP - 10)
        test_start = len(series_df) - n_test
        initial_train_rev = series_df['revenue'].values[:test_start]

        order, aic, trials_df = arima_order_search(initial_train_rev)
        trials_df['cat_id'] = cat
        trials_df['state_id'] = state
        arima_all_trials.append(trials_df)

        arima_best_orders.append({
            'cat_id': cat, 'state_id': state,
            'best_p': order[0], 'best_d': order[1], 'best_q': order[2],
            'AIC': aic, 'n_test_weeks': n_test
        })
        print(f"{cat:10s} {state} -> orde terbaik ARIMA(p,d,q)={order}, AIC={aic:.2f}")

arima_best_orders = pd.DataFrame(arima_best_orders)
arima_tuning_results_all = pd.concat(arima_all_trials, ignore_index=True)

print("\n=== Orde ARIMA Terbaik per Deret (untuk dilampirkan di laporan) ===")
display(arima_best_orders)

print("\n=== Log Lengkap Grid Search ARIMA (semua kombinasi p,d,q per deret) ===")
display(arima_tuning_results_all.sort_values(['cat_id', 'state_id', 'AIC']))

arima_best_orders.to_csv('arima_best_orders.csv', index=False)
arima_tuning_results_all.to_csv('arima_hyperparameter_tuning_log.csv', index=False)
print("\nDisimpan ke: arima_best_orders.csv dan arima_hyperparameter_tuning_log.csv")


### 3. Evaluasi Walk-Forward yang Adil (1D-CNN vs ARIMA)

Kedua model dievaluasi memakai target minggu yang **sama persis** dan histori yang tersedia **sama persis** di setiap langkah (lihat protokol di bagian atas). Metrik yang dihitung: **WRMSSE, MAE, RMSE, MAPE** — sama seperti Tabel 3–6 di paper. Skala WRMSSE (RMS selisih pertama) dihitung dari 10 minggu histori tepat sebelum periode uji, identik untuk kedua model.


In [ ]:
# ==========================================================
# Fungsi Walk-Forward (identik protokolnya untuk CNN & ARIMA)
# ==========================================================
from sklearn.metrics import mean_absolute_error, mean_squared_error

def wrmsse_score(y_true, y_pred, scale_series):
    """RMSSE sederhana: RMSE dibagi skala RMS-selisih-pertama dari `scale_series`
    (10 minggu histori sebelum periode uji -> SAMA untuk CNN & ARIMA)."""
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    diffs = np.diff(scale_series)
    if len(diffs) > 0:
        scale = np.sqrt(np.mean(diffs ** 2))
        if scale > 0:
            return rmse / scale
    return rmse

def build_cnn_from_hp(hp_values, time_step, num_features):
    """Bangun model CNN dari hyperparameter tetap hasil tuning (bukan dari kt.HyperParameters)."""
    inputs = Input(shape=(time_step, num_features))
    x = inputs
    for i in range(hp_values['num_cnn_layers']):
        x = Conv1D(
            filters=hp_values[f'filters_{i}'],
            kernel_size=hp_values[f'kernel_size_{i}'],
            activation='relu',
            padding='causal',
        )(x)
    x = Flatten()(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(hp_values['learning_rate']))
    return model

def walk_forward_cnn(feature_matrix, time_step, n_test, hp_values, epochs=15, batch_size=8):
    """CNN di-retrain ulang (expanding window) di SETIAP langkah walk-forward, sama seperti ARIMA."""
    n = len(feature_matrix)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        train_data = feature_matrix[:i]
        X_train, y_train = create_dataset(train_data, time_step, feature_matrix.shape[1])
        if len(X_train) < 5:
            preds.append(train_data[-1, 0])
            continue
        model = build_cnn_from_hp(hp_values, time_step, feature_matrix.shape[1])
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        X_input = feature_matrix[i - time_step:i].reshape(1, time_step, feature_matrix.shape[1])
        pred = model.predict(X_input, verbose=0)[0, 0]
        preds.append(pred)
    return np.array(preds)

def walk_forward_arima(series_1d, order, n_test):
    """ARIMA di-fit ulang (expanding window) di SETIAP langkah walk-forward, orde tetap."""
    n = len(series_1d)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        hist = series_1d[:i]
        try:
            fc = ARIMA(hist, order=order).fit().forecast(steps=1)[0]
        except Exception:
            fc = hist[-1]
        preds.append(fc)
    return np.array(preds)

print("Fungsi walk-forward CNN & ARIMA siap dipakai.")


In [ ]:
# ==========================================================
# Jalankan Walk-Forward untuk SEMUA kombinasi state x kategori
# (1D-CNN vs ARIMA, protokol identik)
# ==========================================================
import matplotlib.pyplot as plt

walkforward_results = []
walkforward_predictions = {}  # simpan untuk plotting

for cat in CATEGORIES:
    for state in STATES:
        series_df = weekly_avg_enriched[
            (weekly_avg_enriched['cat_id'] == cat) &
            (weekly_avg_enriched['state_id'] == state)
        ].sort_values('wm_yr_wk').reset_index(drop=True)

        if len(series_df) < TIME_STEP + 20:
            print(f"Lewati {cat}-{state}: data terlalu pendek")
            continue

        n_test = min(TEST_WEEKS, len(series_df) - TIME_STEP - 10)
        test_start = len(series_df) - n_test

        # --- ambil orde ARIMA hasil tuning untuk deret ini ---
        order_row = arima_best_orders[
            (arima_best_orders['cat_id'] == cat) & (arima_best_orders['state_id'] == state)
        ]
        if order_row.empty:
            print(f"Lewati {cat}-{state}: orde ARIMA belum di-tuning")
            continue
        order = (int(order_row['best_p'].iloc[0]), int(order_row['best_d'].iloc[0]), int(order_row['best_q'].iloc[0]))

        # --- siapkan fitur untuk CNN (revenue, wday, month), scaled ---
        feat = series_df[['revenue', 'wday', 'month']].values
        scaler = MinMaxScaler()
        feat_scaled = scaler.fit_transform(feat)

        print(f"\n>>> {cat} - {state} | n_test={n_test} minggu | orde ARIMA={order}")

        # --- CNN walk-forward ---
        cnn_pred_scaled = walk_forward_cnn(
            feat_scaled, TIME_STEP, n_test, best_hp_values,
            epochs=CNN_EPOCHS, batch_size=CNN_BATCH
        )
        dummy = np.zeros((len(cnn_pred_scaled), NUM_FEATURES))
        dummy[:, 0] = cnn_pred_scaled
        cnn_pred = scaler.inverse_transform(dummy)[:, 0]

        # --- ARIMA walk-forward ---
        arima_pred = walk_forward_arima(series_df['revenue'].values, order, n_test)

        # --- actual & skala WRMSSE (identik untuk kedua model) ---
        actual = series_df['revenue'].values[test_start:]
        scale_hist = series_df['revenue'].values[test_start - TIME_STEP: test_start]

        walkforward_predictions[(cat, state)] = {
            'actual': actual, 'cnn': cnn_pred, 'arima': arima_pred
        }

        for model_name, pred in [('1D-CNN', cnn_pred), ('ARIMA', arima_pred)]:
            mae = mean_absolute_error(actual, pred)
            rmse = np.sqrt(mean_squared_error(actual, pred))
            mape = np.mean(np.abs((actual - pred) / (actual + 1e-8))) * 100
            wrmsse = wrmsse_score(actual, pred, scale_hist)

            walkforward_results.append({
                'Category': cat, 'State': state, 'Model': model_name,
                'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'WRMSSE': wrmsse
            })
            print(f"  {model_name:7s} -> MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%  WRMSSE={wrmsse:.4f}")

        # --- plot actual vs CNN vs ARIMA ---
        plt.figure(figsize=(10, 4))
        plt.plot(actual, label='Actual', linewidth=2)
        plt.plot(cnn_pred, '--', label='1D-CNN (walk-forward)')
        plt.plot(arima_pred, ':', label='ARIMA (walk-forward)')
        plt.title(f"Walk-Forward Forecast: {cat} - {state}")
        plt.xlabel("Test Week")
        plt.ylabel("Revenue")
        plt.legend()
        plt.tight_layout()
        plt.show()

walkforward_results_df = pd.DataFrame(walkforward_results)
walkforward_results_df.to_csv('walkforward_cnn_vs_arima_results.csv', index=False)
print("\nHasil lengkap disimpan ke: walkforward_cnn_vs_arima_results.csv")
display(walkforward_results_df)


In [ ]:
# ==========================================================
# Tabel Perbandingan Akhir (format sama seperti Tabel 3-6 di paper)
# 1D-CNN vs ARIMA, per metrik, State x Category
# ==========================================================
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

for metric in ['WRMSSE', 'MAE', 'RMSE', 'MAPE']:
    pivot = walkforward_results_df.pivot_table(
        index='State', columns=['Model', 'Category'], values=metric
    )
    # urutkan kolom: Model -> Category sesuai urutan paper
    pivot = pivot.reindex(columns=pd.MultiIndex.from_product(
        [['1D-CNN', 'ARIMA'], ['HOBBIES', 'HOUSEHOLD', 'FOODS']]
    ))
    print(f"\n=== Tabel {metric} — 1D-CNN vs ARIMA (Walk-Forward, Adil) ===")
    display(pivot)

# --- ringkasan rata-rata keseluruhan per model ---
summary = walkforward_results_df.groupby('Model')[['MAE', 'RMSE', 'MAPE', 'WRMSSE']].mean().round(4)
print("\n=== Ringkasan Rata-Rata Keseluruhan (semakin kecil semakin baik) ===")
display(summary)
